# Indexing ClueWeb09 Japanese by OpenSearch for BM25 Model

- [clueweb09/ja](https://ir-datasets.com/clueweb09.html#clueweb09/ja)
- Corpus setup: [dataset/ntcir9-intent1-ja](../../dataset/ntcir9-intent1-ja/README.md)
- Test collections served by this index: `ntcir9-intent1-ja`, `ntcir10-intent2-ja`

67,337,717 Japanese web pages. Text comes from `doc.default_text()`, which
decodes the page (2009-era Japanese pages are commonly Shift_JIS, often
declared only in a `<meta>` tag) and strips tags; its first line is the HTML
title. The full extracted text is indexed with no truncation. Extraction
failures are logged and skipped, never silently dropped.

Unlike the English ClueWeb09-B index, this one needs a **Japanese analyzer**
(kuromoji + ICU) — the default standard analyzer cannot tokenize text without
whitespace. The analyzer below matches the NTCIR-1/2 indexes.

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

The container must have the Japanese analysis plugins installed
(`start_opensearch.sh DATA_DIR --ja`), otherwise index creation fails on the
kuromoji tokenizer.

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

# Confirm the Japanese analysis plugins are present.
plugins = {p["component"] for p in client.cat.plugins(format="json")}
print("kuromoji:", "analysis-kuromoji" in plugins, "| icu:", "analysis-icu" in plugins)

### Index a Corpus for BM25 Model

In [ ]:
import ir_datasets
dataset_name = "clueweb09/ja"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "clueweb09_ja_bm25"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

Japanese analysis

Same configuration as the NTCIR-1/2 indexes, set as the `default` analyzer so
`title` and `text` need no per-field override:

- `icu_normalizer` char filter folds width/case/compatibility variants
- `kuromoji_tokenizer` segments Japanese (search mode also decomposes
  compounds, e.g. 日露戦争 -> 日 / 露 / 戦争; queries tokenize identically)
- `kuromoji_baseform` + `kuromoji_stemmer` normalize inflections
- `kuromoji_part_of_speech` + `ja_stop` drop particles and function words

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      # Disabled during bulk indexing; re-enabled after the run below.
      "refresh_interval": "-1",
      "analysis": {
        "analyzer": {
          "default": {
            "type": "custom",
            "char_filter": ["icu_normalizer"],
            "tokenizer": "kuromoji_tokenizer",
            "filter": [
                "kuromoji_baseform",
                "kuromoji_stemmer",
                "kuromoji_part_of_speech",
                "cjk_width",
                "ja_stop",
            ]
          }
        }
      }
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "url": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

Document preparation

`default_text()` puts the extracted title on the first line; pages that yield
a single line (no newline) have no separate title. Extraction failures land
in `skipped` (log-and-skip).

Note `doc.body` is raw HTML **bytes** — always go through `default_text()`
rather than running string operations on the body.

In [ ]:
skipped = []

def to_action(doc):
    text = doc.default_text()
    nl = text.find("\n")
    title, body = (text[:nl], text[nl+1:]) if nl > 0 else ("", text)
    return {
        "_id": doc.doc_id,
        "_source": {
            "docid": doc.doc_id,
            "url": doc.url,
            "title": title,
            "text": body,
        }
    }

def prepare_documents(dataset):
    for doc in dataset.docs_iter():
        try:
            yield to_action(doc)
        except Exception as e:            # log-and-skip, report at the end
            skipped.append((doc.doc_id, str(e)[:200]))

### Benchmark a slice first

Measured on the first WARC file (31,841 docs, 2026-08): ~1,070 docs/s with
single-threaded extraction -> **~17.5 h** for the full 67.3M docs, and
**4.97 KB/doc of index (~343 GB total)**. Check free disk before starting;
re-run this cell to recalibrate if the hardware changes.

In [ ]:
import itertools, time

N = 10_000
t0 = time.time()
for _ in itertools.islice(prepare_documents(dataset), N):
    pass
rate = N / (time.time() - t0)
print(f"extraction rate: {rate:.0f} docs/s "
      f"-> full corpus lower bound {dataset.docs_count()/rate/3600:.1f} h (excl. indexing overhead)")

Indexing (expect an overnight run)

In [ ]:
from opensearchpy.helpers import parallel_bulk

total = dataset.docs_count()   # 67,337,717

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in parallel_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        chunk_size=500,
        thread_count=4,
        queue_size=4,
        request_timeout=600,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed document
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)},  extraction skipped: {len(skipped)}")
if errors:
    pprint.pprint(errors[:3])
if skipped:
    pprint.pprint(skipped[:10])

# Re-enable refresh now that bulk indexing is done, and make docs searchable.
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}})
client.indices.refresh(index=index_name)
print("final count:", client.count(index=index_name)["count"])

---
### Completeness check

At 67M docs the scan-diff used by the smaller notebooks would hold several GB
of ids in memory; the count comparison below is the default check. Fall back
to the scan-diff (see the msmarco notebooks) only if the counts disagree and
you need the exact missing ids.

In [ ]:
count = client.count(index=index_name)["count"]
expected = dataset.docs_count() - len(skipped)
print(f"index count: {count}  expected: {expected}  missing: {expected - count}")

### Sanity check

Tokenization and a Japanese query against the finished index.

In [ ]:
pprint.pprint([t["token"] for t in client.indices.analyze(
    index=index_name, body={"text": "日露戦争の原因について説明します"})["tokens"]])

resp = client.search(index=index_name, body={
    "size": 5,
    "_source": ["docid", "title"],
    "query": {"multi_match": {"query": "日露戦争", "fields": ["title^2", "text"]}},
})
print("hits:", resp["hits"]["total"]["value"])
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:60]} ({hit['_score']:.2f})")

---
### Notes

- **Near-duplicate filtering:** the Webis CopyCat inclusion lists cover the
  English ClueWebs only; there is no published near-duplicate set for
  ClueWeb09-JA. Deduplication would require running detection (e.g. MinHash)
  over this corpus.
- **Spam filtering:** the Waterloo Fusion spam scores are for the English
  ClueWeb09 and do not apply here.